In [2]:
# Task 3: Hidden Markov Model for Part-of-Speech Tagging

import nltk
from nltk.corpus import brown
from nltk.tag import hmm
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

nltk.download('brown')
nltk.download('universal_tagset')

print("Resources loaded successfully!")

Resources loaded successfully!


[nltk_data] Downloading package brown to /home/parth/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package universal_tagset to
[nltk_data]     /home/parth/nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!


In [3]:
# (a) Load the Brown Corpus with universal POS tags
tagged_sents = brown.tagged_sents(categories='news', tagset='universal')

print(f"Total tagged sentences in news category: {len(tagged_sents)}")
print("\nSample tagged sentence (first 7 words):")
print(tagged_sents[0][:7])

Total tagged sentences in news category: 4623

Sample tagged sentence (first 7 words):
[('The', 'DET'), ('Fulton', 'NOUN'), ('County', 'NOUN'), ('Grand', 'ADJ'), ('Jury', 'NOUN'), ('said', 'VERB'), ('Friday', 'NOUN')]


In [4]:
# (b) Divide tagged sentences into training and testing sets (80% train, 20% test)
split_idx = int(len(tagged_sents) * 0.8)
train_sents = tagged_sents[:split_idx]
test_sents = tagged_sents[split_idx:]

print(f"Training sentences: {len(train_sents)}")
print(f"Testing sentences:  {len(test_sents)}")

Training sentences: 3698
Testing sentences:  925


In [5]:
# (c) Train an HMM-based tagger
trainer = hmm.HiddenMarkovModelTrainer()
hmm_tagger = trainer.train_supervised(train_sents)

print("HMM Tagger training completed successfully!")

HMM Tagger training completed successfully!


In [6]:
# (d) Predict tags for at least five unseen sentences
test_sentences = [
    "The government announced a new policy today",
    "He will visit the hospital tomorrow morning",
    "She read an interesting book about history",
    "They walked through the quiet park quickly",
    "The company reported higher profits this year"
]

predictions = []
for sent in test_sentences:
    words = sent.split()
    tagged = hmm_tagger.tag(words)
    predictions.append({
        "Sentence": sent,
        "Predicted POS Tags": " ".join([f"{w}/{t}" for w, t in tagged])
    })

pred_df = pd.DataFrame(predictions)
display(pred_df)

,Sentence,Predicted POS Tags
0,The government announced a new policy today,The/DET government/NOUN announced/VERB a/DET n...
1,He will visit the hospital tomorrow morning,He/PRON will/VERB visit/VERB the/DET hospital/...
2,She read an interesting book about history,She/PRON read/VERB an/DET interesting/ADJ book...
3,They walked through the quiet park quickly,They/PRON walked/VERB through/ADP the/DET quie...
4,The company reported higher profits this year,The/DET company/NOUN reported/VERB higher/ADJ ...


In [7]:
# (e) Calculate test accuracy
test_accuracy = hmm_tagger.accuracy(test_sents[:200])
print(f"HMM Test Accuracy on unseen test sentences: {test_accuracy * 100:.2f}%")

HMM Test Accuracy on unseen test sentences: 31.92%


## Observations

### (f) Explanation of HMM Components in Sequence Tagging

An HMM-based Part-of-Speech tagger operates using four fundamental statistical components:

1. **Hidden States ($S$):** The underlying POS tags (`NOUN`, `VERB`, `DET`, `ADJ`, etc.). They are "hidden" because they are not visible in raw text; only the words are observed.
2. **Observations ($O$):** The sequence of actual surface words emitted by the hidden states (e.g., *"The"*, *"government"*, *"announced"*).
3. **Transition Probabilities ($A$):** The probability of moving from one POS tag to another:
   $$P(t_i \mid t_{i-1}) = \frac{\text{Count}(t_{i-1}, t_i)}{\text{Count}(t_{i-1})}$$
   For example, $P(\text{NOUN} \mid \text{DET})$ is high, whereas $P(\text{DET} \mid \text{DET})$ is virtually zero.
4. **Emission Probabilities ($B$):** The probability of a specific POS tag generating a specific word:
   $$P(w_i \mid t_i) = \frac{\text{Count}(t_i, w_i)}{\text{Count}(t_i)}$$
   For example, the tag `VERB` has a high emission probability for words like *"announced"* or *"read"*.

The Viterbi algorithm uses these transition and emission probabilities to find the most probable sequence of POS tags for any unseen input sentence.